# Synthetic Data Generation for LLM Evaluation

In this tutorial, we build a configurable synthetic data generator for LLM evaluation.

Instead of writing one prompt for one dataset, we create functions that can be called with different parameters:
- categories and their distribution
- question length distribution
- personas
- generation strategy

We use the same customer support setting as in previous tutorials. The output is a dataset of customer questions and reference answers that can later be used for correctness evaluation, judge calibration, or regression testing.

The goal is not to fully automate dataset creation, but to create a controllable first draft that humans can review and refine.

## 1. Setup

We use `pandas` for dataset processing and Pydantic models for structured generation outputs.

In [1]:
import os
import random
from typing import Any, Dict, List, Literal, Optional

import pandas as pd
from pydantic import BaseModel

pd.set_option("display.max_colwidth", None)

## 2. Output schema

The generator returns structured examples.

Using a schema makes the output easier to validate and convert into a DataFrame.

In [2]:
class SyntheticExample(BaseModel):
    generation_strategy: str
    category: str
    customer_question: str
    reference_answer: str
    question_length: Literal["short", "medium", "long"]
    persona: Optional[str] = None
    edge_case_type: Optional[str] = None
    policy_fact_used: Optional[str] = None


class SyntheticDataset(BaseModel):
    examples: List[SyntheticExample]

## 3. Demo generator

The notebook can run without an API key.

For this case, we use a small demo generator that follows the same parameters.

In [3]:
QUESTION_TEMPLATES = {
    "returns": {
        "short": [
            "Can I return an item if it does not fit?",
            "Can I return a gift?",
        ],
        "medium": [
            "I ordered a jacket, but the size is wrong. The tags are still attached and I have not worn it. Can I return it?",
            "I received an item as a gift and I do not have the original receipt. I can provide the order number if needed. Can I still return it?",
        ],
        "long": [
            "I bought a pair of shoes a few weeks ago, but they do not fit well. I only tried them on indoors and still have the original box. I am not sure if I am still within the return window. Can I return them or exchange them for another size?",
            "I ordered two items and want to return only one of them. The item is unused, but I opened the packaging to check the size. I still have the confirmation email and order number. What should I check before starting a return?",
        ],
    },
    "shipping": {
        "short": [
            "How long does shipping usually take?",
            "Do you offer international shipping?",
        ],
        "medium": [
            "I need my order before the weekend. I am not sure which shipping method I selected. How can I check the estimated delivery date?",
            "I want to place an order from another country. The checkout page shows several shipping options. How can I know whether delivery is available to my address?",
        ],
        "long": [
            "I placed an order yesterday and I am trying to understand when it might arrive. The confirmation email has a shipping method, but I do not know whether the estimate is guaranteed. What should I look at before contacting support?",
            "I am buying a gift for someone in another country and want to know whether it can arrive on time. I see different delivery options at checkout, but I am not sure if international delivery is available for every item. How should I think about shipping estimates?",
        ],
    },
    "payments": {
        "short": [
            "Why was my payment declined?",
            "Do you accept gift cards?",
        ],
        "medium": [
            "I tried to pay with my card, but the payment did not go through. The card works elsewhere. What should I try next?",
            "I have a gift card and a credit card. I want to use both for one order. Is that usually possible?",
        ],
        "long": [
            "I tried to place an order twice, but both payment attempts failed. I checked that the billing address is correct and the card has enough funds. I do not know whether the issue is with the store, my bank, or the payment provider. What should I do?",
            "I received a gift card and want to use it for part of my order. The total is higher than the gift card balance, so I may need another payment method too. What should I check before placing the order?",
        ],
    },
    "refunds": {
        "short": [
            "When will I receive my refund?",
            "Where will my refund be sent?",
        ],
        "medium": [
            "I returned an item last week and have not seen the refund yet. I used a credit card for the purchase. How long does this usually take?",
            "My item arrived damaged and I want a refund. I have photos of the damage and the packaging. What should I do next?",
        ],
        "long": [
            "I returned an item after receiving approval from support, but I have not received my refund yet. The tracking says the package arrived at the warehouse several days ago. I paid with my original credit card. What steps should I take before escalating this?",
            "I received a damaged item and want to understand whether I can get a refund or replacement. I still have the packaging and can take photos. I am not sure whether I should return the item first or contact support. What is the safest next step?",
        ],
    },
    "tracking": {
        "short": [
            "How can I track my order?",
            "Where is my package?",
        ],
        "medium": [
            "The tracking link in my email has not updated for two days. The estimated delivery date has not passed yet. Should I contact support now?",
            "I cannot find my tracking number, but I have the order confirmation email. Where should I look first?",
        ],
        "long": [
            "I placed an order last week and received a confirmation email, but I cannot find a separate shipping email. I checked my account page and still do not see a tracking link. I want to know whether the order has shipped. What should I do next?",
            "The carrier tracking page says my package is in transit, but the delivery date changed twice. I need the item soon and I am not sure whether this is normal. What information should I check before contacting customer support?",
        ],
    },
    "cancellations": {
        "short": [
            "Can I cancel my order?",
            "Can I cancel my subscription?",
        ],
        "medium": [
            "I placed an order by mistake a few minutes ago. I do not think it has shipped yet. Can I still cancel it?",
            "I want to cancel my subscription before the next billing date. Where should I look in my account?",
        ],
        "long": [
            "I accidentally placed an order with the wrong item and noticed the mistake right away. I have not received a shipping confirmation yet. I want to cancel it before it is processed. What should I do as quickly as possible?",
            "I signed up for a subscription but no longer want the next renewal. I am not sure where the cancellation option is located in my account. I also want to avoid being charged again. What should I check?",
        ],
    },
}

REFERENCE_TEMPLATES = {
    "returns": "Return eligibility depends on the store policy, return window, item condition, and proof of purchase.",
    "shipping": "Shipping time depends on the destination, carrier, selected shipping method, and checkout estimate.",
    "payments": "Payment issues may come from the bank, card issuer, billing details, or payment provider.",
    "refunds": "Refund timing and eligibility depend on the store policy, item status, and payment provider processing time.",
    "tracking": "Customers can usually track orders through the confirmation email, shipping email, or account page.",
    "cancellations": "Cancellation may be possible if the order or subscription has not already been processed, shipped, or renewed.",
}

EDGE_CASE_TYPES = [
    "ambiguous request",
    "missing information",
    "conflicting request",
    "risky policy assumption",
    "emotional customer",
    "multi-intent request",
]


def sample_from_distribution(distribution: Dict[str, float]) -> str:
    values = list(distribution.keys())
    weights = list(distribution.values())
    return random.choices(values, weights=weights, k=1)[0]


def demo_generate_examples(
    *,
    n_examples: int,
    strategy: str,
    category_distribution: Dict[str, float],
    question_length_distribution: Dict[str, float],
    personas: Optional[List[str]] = None,
    policy_text: Optional[str] = None,
) -> List[Dict[str, Any]]:
    rows = []

    for _ in range(n_examples):
        category = sample_from_distribution(category_distribution)
        question_length = sample_from_distribution(question_length_distribution)
        persona = random.choice(personas) if personas else None

        question = random.choice(QUESTION_TEMPLATES[category][question_length])
        reference_answer = REFERENCE_TEMPLATES[category]

        edge_case_type = None
        policy_fact_used = None

        if strategy == "edge_case":
            edge_case_type = random.choice(EDGE_CASE_TYPES)
            reference_answer = f"{reference_answer} Ask for missing details when needed and avoid unsupported guarantees."

        if strategy == "policy_based":
            policy_fact_used = "Use only the provided policy facts."
            reference_answer = f"{reference_answer} The answer must be grounded only in the provided policy."

        if strategy == "persona_based" and persona:
            question = f"[{persona}] {question}"
            reference_answer = f"{reference_answer} Keep the response appropriate for a {persona}."

        rows.append({
            "generation_strategy": strategy,
            "category": category,
            "customer_question": question,
            "reference_answer": reference_answer,
            "question_length": question_length,
            "persona": persona,
            "edge_case_type": edge_case_type,
            "policy_fact_used": policy_fact_used,
        })

    return rows

## 4. Prompt builder

The prompt is built from external parameters.

This makes generation easier to control and reuse.

In [4]:
def format_distribution(distribution: Dict[str, float]) -> str:
    return "\n".join(
        [f"- {key}: {value}" for key, value in distribution.items()]
    )


def build_generation_prompt(
    *,
    n_examples: int,
    strategy: Literal["basic", "edge_case", "policy_based", "persona_based"],
    category_distribution: Dict[str, float],
    question_length_distribution: Dict[str, float],
    personas: Optional[List[str]] = None,
    policy_text: Optional[str] = None,
    diversity_hint: Optional[str] = None,
) -> str:
    persona_block = "No personas."
    if personas:
        persona_block = "\n".join([f"- {persona}" for persona in personas])

    policy_block = "No policy text provided."
    if policy_text:
        policy_block = policy_text

    return f"""Generate {n_examples} synthetic evaluation examples for an e-commerce customer support assistant.

Generation strategy:
{strategy}

Category distribution:
{format_distribution(category_distribution)}

Question length distribution:
{format_distribution(question_length_distribution)}

Question length definitions:
- short: 1-2 sentences
- medium: 3-4 sentences
- long: 5 or more sentences

Personas:
{persona_block}

Policy text:
{policy_block}

Diversity hint:
{diversity_hint or "Create varied examples. Avoid repeating wording across examples."}

Rules:
- Generate customer questions and reference answers.
- Follow the requested category and question length distributions approximately.
- Reference answers should be concise, safe, and useful.
- Avoid unsupported store-specific claims unless they are grounded in the policy text.
- For edge_case strategy, include edge_case_type.
- For policy_based strategy, use only the provided policy and include policy_fact_used.
- For persona_based strategy, use the provided personas.
- The question_length field must describe the length of customer_question, not reference_answer.

Return structured data matching the schema.
"""

## 5. Configurable generator function

This is the main function.

Calling it multiple times can produce different examples because the prompt includes a diversity hint and the model runs with a non-zero temperature.

In [5]:
def generate_synthetic_dataset(
    *,
    n_examples: int,
    strategy: Literal["basic", "edge_case", "policy_based", "persona_based"],
    category_distribution: Dict[str, float],
    question_length_distribution: Dict[str, float],
    personas: Optional[List[str]] = None,
    policy_text: Optional[str] = None,
    model: str = "gpt-4.1-mini",
    temperature: float = 0.8,
    diversity_hint: Optional[str] = None,
    use_api: Optional[bool] = None,
) -> pd.DataFrame:
    if use_api is None:
        use_api = bool(os.getenv("OPENAI_API_KEY"))

    if not use_api:
        examples = demo_generate_examples(
            n_examples=n_examples,
            strategy=strategy,
            category_distribution=category_distribution,
            question_length_distribution=question_length_distribution,
            personas=personas,
            policy_text=policy_text,
        )
        return pd.DataFrame(examples)

    from openai import OpenAI

    client = OpenAI()

    prompt = build_generation_prompt(
        n_examples=n_examples,
        strategy=strategy,
        category_distribution=category_distribution,
        question_length_distribution=question_length_distribution,
        personas=personas,
        policy_text=policy_text,
        diversity_hint=diversity_hint,
    )

    response = client.responses.parse(
        model=model,
        input=[
            {
                "role": "system",
                "content": "You generate compact synthetic datasets for LLM evaluation.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        text_format=SyntheticDataset,
        temperature=temperature,
    )

    parsed = response.output_parsed
    return pd.DataFrame([example.model_dump() for example in parsed.examples])

## 6. Strategy wrappers

Wrapper functions make common generation strategies easier to call.

In [6]:
def generate_basic_examples(**kwargs) -> pd.DataFrame:
    return generate_synthetic_dataset(strategy="basic", **kwargs)


def generate_edge_cases(**kwargs) -> pd.DataFrame:
    return generate_synthetic_dataset(strategy="edge_case", **kwargs)


def generate_policy_based_examples(policy_text: str, **kwargs) -> pd.DataFrame:
    return generate_synthetic_dataset(
        strategy="policy_based",
        policy_text=policy_text,
        **kwargs,
    )


def generate_persona_based_examples(personas: List[str], **kwargs) -> pd.DataFrame:
    return generate_synthetic_dataset(
        strategy="persona_based",
        personas=personas,
        **kwargs,
    )

## 7. Parameters

Now we define category and question length distributions externally.

In [7]:
category_distribution = {
    "returns": 0.30,
    "shipping": 0.25,
    "payments": 0.15,
    "refunds": 0.15,
    "tracking": 0.10,
    "cancellations": 0.05,
}

question_length_distribution = {
    "short": 0.50,
    "medium": 0.35,
    "long": 0.15,
}

personas = [
    "angry customer",
    "confused first-time buyer",
    "impatient customer",
    "international customer",
    "budget-conscious customer",
    "anxious customer",
]

policy_text = """
Return policy:
- Most items can be returned within 30 days of delivery.
- Items must be unused and in original packaging.
- Final sale items cannot be returned.
- Damaged items should be reported within 7 days of delivery with photos.

Shipping policy:
- Standard shipping usually takes 3-7 business days.
- Express shipping usually takes 1-3 business days.
- Shipping estimates are not guarantees.
"""

## 8. Generate basic examples

We start with a simple dataset controlled by category and question length distributions.

In [8]:
basic_df = generate_basic_examples(
    n_examples=8,
    category_distribution=category_distribution,
    question_length_distribution=question_length_distribution,
    diversity_hint="Focus on common customer support questions.",
)

basic_df

,generation_strategy,category,customer_question,reference_answer,question_length,persona,edge_case_type,policy_fact_used
0,basic,returns,Can I return a product if I don't like it?,"Yes, you can return products within our return period as long as they are in original condition.",short,None,None,None
1,basic,shipping,How long does shipping usually take?,Shipping typically takes 3-5 business days depending on your location.,short,None,None,None
2,basic,payments,I tried to pay with my credit card but it was declined. What should I do?,"Please check with your card provider to ensure your card is active and has sufficient funds, then try again.",medium,None,None,None
3,basic,refunds,When will I get my refund after returning an item?,Refunds are usually processed within 7-10 business days after we receive the returned item.,short,None,None,None
4,basic,shipping,I placed my order last week but haven't received any shipping updates. Can you tell me when it will be shipped?,"Orders are usually shipped within 1-2 business days. If you haven't received any update, please provide your order number and we'll check the status.",medium,None,None,None
5,basic,tracking,How can I track my package?,You can track your package using the tracking link sent to your email after shipment.,short,None,None,None
6,basic,returns,I received a damaged item and want to return it. What steps should I follow to ensure the return is accepted?,Please contact customer support with photos of the damaged item to start the return process.,medium,None,None,None
7,basic,cancellations,"I just placed an order but realized I don't need it anymore. Can I cancel it? If so, how do I proceed and are there any fees?",You can cancel your order within 1 hour of placing it without any fees. Please contact support immediately to request cancellation.,long,None,None,None


## 9. Generate edge cases

Now we use the same distributions, but ask for harder examples.

In [9]:
edge_df = generate_edge_cases(
    n_examples=8,
    category_distribution=category_distribution,
    question_length_distribution=question_length_distribution,
    diversity_hint="Focus on ambiguous, emotional, or policy-sensitive questions.",
)

edge_df

,generation_strategy,category,customer_question,reference_answer,question_length,persona,edge_case_type,policy_fact_used
0,edge_case,returns,Can I return an item if I lost the receipt but still have the packaging?,Returns usually require proof of purchase like a receipt. Please contact customer support to explore possible options.,short,None,missing_receipt,None
1,edge_case,shipping,"My order was supposed to arrive yesterday, but I haven't received any updates. Can you check if it got lost or delayed?",I understand your concern. I'll check the shipment status and update you shortly.,medium,None,missing_update,None
2,edge_case,payments,"I was charged twice for the same order, but only one order confirmation email was sent. What should I do?",Double charges can occur due to processing errors. Please provide your order details so we can investigate and resolve this.,medium,None,duplicate_charge,None
3,edge_case,refunds,"It's been over a month since I returned my product, and I haven't received my refund yet. Is there a problem?",Refunds typically take a certain number of days. I'll check your case and update you on the status promptly.,medium,None,delayed_refund,None
4,edge_case,tracking,"The tracking link shows my package was delivered, but I never received it and no one signed for it. What can I do?","If your package shows delivered but you haven’t received it, please check with neighbors or your local delivery service. Contact support if it remains missing.",medium,None,delivery_discrepancy,None
5,edge_case,returns,I want to return a gift item without the original packaging. Is that possible?,Returns generally require original packaging. Please contact customer support for assistance with gift returns.,short,None,missing_packaging,None
6,edge_case,cancellations,I tried cancelling my order after it was shipped. Is there any way to stop delivery or return it after arrival?,"Orders cannot be cancelled once shipped, but you can return the item after delivery according to our returns policy.",long,None,late_cancellation_attempt,None
7,edge_case,shipping,"My address changed after placing the order, but I didn't update it in time. Will the package be delivered to my old address or can it be rerouted?","Once shipped, address changes are difficult. Please contact the carrier immediately to request rerouting, and notify us if you need further help.",long,None,address_change_post_shipment,None


## 10. Generate policy-based examples

Policy-based examples are useful for reference-based correctness evaluation.

In [10]:
policy_df = generate_policy_based_examples(
    policy_text=policy_text,
    n_examples=8,
    category_distribution={
        "returns": 0.50,
        "shipping": 0.50,
    },
    question_length_distribution={
        "short": 0.25,
        "medium": 0.50,
        "long": 0.25,
    },
    diversity_hint="Create examples that test exact policy boundaries.",
)

policy_df

,generation_strategy,category,customer_question,reference_answer,question_length,persona,edge_case_type,policy_fact_used
0,policy_based,returns,Can I return an item if it has been opened but not used?,Items must be unused and in original packaging to be returned.,short,None,opened_but_unused_item,Items must be unused and in original packaging.
1,policy_based,shipping,How long does standard shipping usually take?,Standard shipping usually takes 3-7 business days.,short,None,None,Standard shipping usually takes 3-7 business days.
2,policy_based,returns,I bought a final sale jacket last week but changed my mind. Can I return it?,Final sale items cannot be returned.,medium,None,final_sale_item,Final sale items cannot be returned.
3,policy_based,shipping,"If I choose express shipping, how soon should I expect my package to arrive?",Express shipping usually takes 1-3 business days.,medium,None,None,Express shipping usually takes 1-3 business days.
4,policy_based,returns,"I received a damaged blender yesterday, but I’m just now noticing it. Can I still report it and get a replacement?",Damaged items should be reported within 7 days of delivery with photos.,long,None,late_damage_report,Damaged items should be reported within 7 days of delivery with photos.
5,policy_based,shipping,"I placed an order and selected standard shipping. The website says 3-7 business days, but will it definitely arrive by then?",Shipping estimates are not guarantees; actual delivery may vary.,long,None,shipping_estimate_expectation,Shipping estimates are not guarantees.
6,policy_based,returns,My package arrived 31 days ago and I just realized the item isn’t what I expected. Can I still return it?,Most items can be returned within 30 days of delivery.,medium,None,return_after_30_days,Most items can be returned within 30 days of delivery.
7,policy_based,shipping,Does express shipping guarantee delivery within 3 business days even during holidays?,"Shipping estimates are not guarantees, so delivery times may vary including during holidays.",medium,None,holiday_shipping_guarantee,Shipping estimates are not guarantees.


## 11. Generate persona-based examples

Personas help test whether reference answers remain helpful across different user styles.

In [11]:
persona_df = generate_persona_based_examples(
    personas=personas,
    n_examples=8,
    category_distribution=category_distribution,
    question_length_distribution=question_length_distribution,
    diversity_hint="Use different user emotions and levels of detail.",
)

persona_df

,generation_strategy,category,customer_question,reference_answer,question_length,persona,edge_case_type,policy_fact_used
0,persona_based,returns,"I received the wrong size, and I want to send it back immediately. What do I do?","You can start a return through your order page. Make sure the item is unused and in original packaging. Once we receive it, we will process your refund.",short,angry customer,None,None
1,persona_based,shipping,I'm a bit confused about how long shipping will take. Can you tell me when I might get my order?,Standard shipping usually takes 5-7 business days. You can check your order status in your account for updates.,short,confused first-time buyer,None,None
2,persona_based,payments,I tried paying with my credit card but got an error. I’m not sure if it’s because my card isn’t accepted or something else. What should I do next?,"Please check that your card details are correct and that your card supports online payments. If the problem continues, try another payment method or contact your bank.",medium,impatient customer,None,None
3,persona_based,refunds,I returned an item last week and haven’t received my refund yet. Can you help me understand when it will be processed?,Refunds are usually processed within 5-10 business days after we receive the returned item. Please check your order status for updates.,medium,anxious customer,None,None
4,persona_based,tracking,I live outside the country and I’m worried about tracking my package. Is there a way to get updates internationally?,"Yes, international shipments include tracking. You can use the tracking number provided in your order confirmation to monitor the package through the carrier’s website.",medium,international customer,None,None
5,persona_based,returns,I want to return two items but one is opened and used. Can I still return both together?,"Only unopened and unused items are eligible for return. You can return the unopened item, but the used one cannot be accepted.",medium,budget-conscious customer,partial returns,None
6,persona_based,shipping,It’s been over two weeks and my order still hasn’t arrived. I need this urgently. What can you do?,I’m sorry for the delay. Please provide your order number so we can check its status and assist you further.,short,angry customer,None,None
7,persona_based,cancellations,I placed an order yesterday but changed my mind. Can I cancel it now and get my money back? How do cancellations work?,You can cancel your order if it hasn't shipped yet. Please contact support as soon as possible to request cancellation and refund.,long,confused first-time buyer,None,None


## 12. Generate multiple batches

The same function can be called multiple times to expand the dataset.

With API calls, non-zero temperature and different diversity hints help produce different examples.

In [12]:
batch_1 = generate_basic_examples(
    n_examples=5,
    category_distribution=category_distribution,
    question_length_distribution=question_length_distribution,
    diversity_hint="Focus on common pre-purchase questions.",
)

batch_2 = generate_basic_examples(
    n_examples=5,
    category_distribution=category_distribution,
    question_length_distribution=question_length_distribution,
    diversity_hint="Focus on post-purchase support questions.",
)

multi_batch_df = pd.concat([batch_1, batch_2], ignore_index=True)

multi_batch_df

,generation_strategy,category,customer_question,reference_answer,question_length,persona,edge_case_type,policy_fact_used
0,basic,returns,Can I return a product if I don't like it?,"Yes, you can return most products within the return period if you are not satisfied. Please check our return policy for specific conditions.",short,None,None,None
1,basic,shipping,How long does shipping usually take for standard delivery?,Standard shipping typically takes 5-7 business days depending on your location.,short,None,None,None
2,basic,payments,I want to buy a gift card but I'm not sure which payment options are available. Could you tell me what payment methods you accept for purchasing gift cards?,"We accept major credit cards, debit cards, and PayPal for gift card purchases.",medium,None,None,None
3,basic,refunds,I returned an item last week but haven't received my refund yet. How long does it usually take for refunds to be processed after you receive the returned product?,Refunds are usually processed within 5-10 business days after we receive your returned item.,medium,None,None,None
4,basic,tracking,I placed an order yesterday and would like to know how I can track its delivery status. Can you explain the steps to check my shipment tracking information?,"Once your order ships, we will send you a tracking number via email. You can use this number on the carrier's website to check your shipment status.",long,None,None,None
5,basic,returns,Can I return an item I bought last week?,"Yes, you can return most items within the return period. Please check the return policy for specific details.",short,None,None,None
6,basic,shipping,"My package hasn't arrived yet, and it's been over a10 days since I ordered. What can I do?","We recommend checking the tracking information first. If the package is delayed beyond the estimated delivery time, please contact customer support for assistance.",medium,None,None,None
7,basic,refunds,I returned a defective product two weeks ago but haven’t received my refund yet. How long does it usually take?,"Refunds typically take 7-10 business days after we receive the returned item. If it has been longer, please contact support to check the status.",medium,None,None,None
8,basic,payments,Is it safe to save my credit card information on your website for faster checkout?,"Yes, we use secure encryption methods to protect your payment information when you save it on our site.",short,None,None,None
9,basic,shipping,I need to cancel my order because I entered the wrong shipping address. What steps should I follow to update or cancel my order?,"If your order hasn’t shipped yet, you can cancel it from your account. Otherwise, please contact customer support to update the shipping address or arrange a return after delivery.",long,None,None,None


## 13. Combine generated datasets

We can combine all strategies into one evaluation dataset.

In [13]:
all_examples_df = pd.concat(
    [
        basic_df,
        edge_df,
        policy_df,
        persona_df,
        multi_batch_df,
    ],
    ignore_index=True,
)

all_examples_df

,generation_strategy,category,customer_question,reference_answer,question_length,persona,edge_case_type,policy_fact_used
0,basic,returns,Can I return a product if I don't like it?,"Yes, you can return products within our return period as long as they are in original condition.",short,None,None,None
1,basic,shipping,How long does shipping usually take?,Shipping typically takes 3-5 business days depending on your location.,short,None,None,None
2,basic,payments,I tried to pay with my credit card but it was declined. What should I do?,"Please check with your card provider to ensure your card is active and has sufficient funds, then try again.",medium,None,None,None
3,basic,refunds,When will I get my refund after returning an item?,Refunds are usually processed within 7-10 business days after we receive the returned item.,short,None,None,None
4,basic,shipping,I placed my order last week but haven't received any shipping updates. Can you tell me when it will be shipped?,"Orders are usually shipped within 1-2 business days. If you haven't received any update, please provide your order number and we'll check the status.",medium,None,None,None
5,basic,tracking,How can I track my package?,You can track your package using the tracking link sent to your email after shipment.,short,None,None,None
6,basic,returns,I received a damaged item and want to return it. What steps should I follow to ensure the return is accepted?,Please contact customer support with photos of the damaged item to start the return process.,medium,None,None,None
7,basic,cancellations,"I just placed an order but realized I don't need it anymore. Can I cancel it? If so, how do I proceed and are there any fees?",You can cancel your order within 1 hour of placing it without any fees. Please contact support immediately to request cancellation.,long,None,None,None
8,edge_case,returns,Can I return an item if I lost the receipt but still have the packaging?,Returns usually require proof of purchase like a receipt. Please contact customer support to explore possible options.,short,None,missing_receipt,None
9,edge_case,shipping,"My order was supposed to arrive yesterday, but I haven't received any updates. Can you check if it got lost or delayed?",I understand your concern. I'll check the shipment status and update you shortly.,medium,None,missing_update,None


## 14. Coverage summary

Always inspect coverage before using synthetic data for evaluation.

In [14]:
strategy_counts = all_examples_df["generation_strategy"].value_counts()
category_counts = all_examples_df["category"].value_counts()
question_length_counts = all_examples_df["question_length"].value_counts()

display(strategy_counts)
display(category_counts)
display(question_length_counts)

generation_strategy
basic            18
edge_case         8
policy_based      8
persona_based     8
Name: count, dtype: int64

category
shipping         13
returns          12
payments          5
refunds           5
tracking          4
cancellations     3
Name: count, dtype: int64

question_length
medium    19
short     15
long       8
Name: count, dtype: int64

## 15. Export the dataset

The generated dataset can be reviewed by humans and reused in later evaluation workflows.

In [15]:
output_path = "synthetic_evaluation_dataset.csv"

all_examples_df.to_csv(output_path, index=False)

output_path

'synthetic_evaluation_dataset.csv'

## 16. Takeaways

Synthetic data generation becomes more useful when it is configurable.

By controlling categories, distributions, question length, personas, and strategy, we can create datasets that better match the evaluation goal.

The generated examples should still be reviewed before they become a trusted evaluation dataset.